In [ ]:
import os
import certifi
import requests
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests

In [16]:
from langchain.agents import create_react_agent, AgentExecutor

In [ ]:
# ======================
#   load env variables
# ======================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [18]:
search_tool = TavilySearchResults(max_results=2)

In [ ]:
# CReate our custom tool to fetch weather data using the Weatherstack API

@tool
def get_weather_data(city: str) -> str:
    """
    Fetches weather data for a given city.
    """
    url=(
        f"https://weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()
    if "current" not in data:
        return f"Could not fetch weather data for {city}."
    return(
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C,\n "
        f"Description: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%,\n"
    )

In [19]:
result = search_tool.invoke("Give me the latest news on AI")
result

[{'url': 'https://guides.library.georgetown.edu/ai/news',
  'content': "Privatized air traffic control would mean safer skies This link opens in a new window_by The Washington Post 20 Oct 2026Formats: Citation,Full text,Find Full Text @ GU,Find Full Text @ GU_ Oct 19, 2026 \n   The U.S. and China need an arms control pact — for AI This link opens in a new window_by Asha Jadeja The Washington Post 20 Oct 2026Formats: Citation,Full text,Find Full Text @ GU,Find Full Text @ GU_ Oct 19, 2026 \n   Elon Musk Beats Anthropic on Giveaways; He says he'll give every child a share of SpaceX, aka a stake in capitalism.This link opens in a new window_Wall Street Journal (Online) 19 Oct 2026Formats: Citation,Full text,Find Full Text @ GU,Find Full Text @ GU_ Oct 19, 2026 [...] DHS Orders DeepSeek Technology Removed From Federal Civilian IT Systems; The order also bars future purchases of the company's products and services This link opens in a new window_Hart, Connor Wall Street Journal (Online) 19 

In [20]:
# ======================
# LLM
# =====================

llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0,
    api_key=GOOGLE_API_KEY,
)


In [22]:
response =llm.invoke("What position does CR7 play?")
response

AIMessage(content='Cristiano Ronaldo (CR7) currently plays as a **Striker / Centre-Forward**. \n\nThroughout his career, his position has evolved:\n\n* **Current role:** **Centre-Forward / Striker (ST/CF)** – In the latter stages of his career (at Juventus, his second stint at Manchester United, Al-Nassr, and for Portugal), he transitioned into a central attacking role focused on finishing inside the penalty area.\n* **Prime role:** **Left Winger / Inside Forward (LW)** – During the majority of his time at Real Madrid and his peak years, he played primarily on the left wing, cutting inside on his stronger right foot to score.\n* **Early career:** **Right/Left Winger (RW/LW)** – At Sporting CP and his early years at Manchester United, he played as a traditional, highly-skilled winger delivering crosses and taking on defenders on the flanks.', response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-bec43925-

In [23]:
# PROMPT
# ======================

prompt = hub.pull("hwchase17/react")

c:\Users\farsi\.conda\envs\langagent\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [24]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [25]:
tools = [search_tool]

In [26]:
# ======================
# create agent
# ======================

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
 
)

In [27]:
# ======================
# Executor
# ======================    
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)  

In [ ]:
# ======================
# RUN
# ====================
response = agent_executor.invoke({ 
    "input": ("Find the capital of India")
})    



> Entering new AgentExecutor chain...


Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.
Please retry in 6h18m44.607992243s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  description: "Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits."
  quota_metric: "generativelanguage.googleapis

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.
Please retry in 6h13m40.909039011s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  description: "Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits."
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_d